In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

In [ ]:
ratings = pd.read_csv('data/ratings.csv')
movies = pd.read_csv('data/movies.csv')
ratings.shape, movies.shape

In [ ]:
ratings.head()

In [ ]:
# genres, one row per movie-genre
genres = movies.assign(genre=movies.genres.str.split('|')).explode('genre')
genre_list = list(set(genres.genre))
len(genre_list)

In [ ]:
stats = ratings.groupby('movieId').agg(n=('rating', 'size'), mean=('rating', 'mean')).reset_index()
popular = stats[stats.n >= 50].merge(movies, on='movieId')
top10 = popular.sort_values('mean', ascending=False).head(10)
top10[['title', 'n', 'mean']]

In [ ]:
fig, ax = plt.subplots()
ratings.rating.hist(bins=10, ax=ax)
ax.set_title('Ratings')
plt.show()

In [ ]:
user_feats = ratings.groupby('userId').rating.agg(['mean', 'count', 'std']).fillna(0)
X = ratings.merge(user_feats, left_on='userId', right_index=True)[['mean', 'count', 'std']].sample(20000)
y = ratings.loc[X.index, 'rating']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
model = RandomForestRegressor(n_estimators=30, n_jobs=-1)
model.fit(X_train, y_train)
pred = model.predict(X_test)

In [ ]:
mae = mean_absolute_error(y_test, pred)
print(f'MAE: {mae:.4f}')